## Ход выполнения работы

1. Исходный репозиторий: `tylerrussin/Fish-Dimensions-Regression-Analysis`, исходный коммит `1f65547b63c3043e2dcfff0c6435afa993331d4e`; рабочая ветка `lab/01-eda`.
2. Для запуска проекта указана среда Python 3.13; в `Pipfile` обновлена версия Python, в `run.py` заменён устаревший `app.run_server()` на `app.run()`, в `app.py` удалён несовместимый аргумент `light`.
3. Исследовательский блокнот `notebooks/01-eda.ipynb` читает `assets/data/Fish.csv` относительно найденного корня проекта, поэтому не зависит от текущей папки Jupyter.
4. Повторно используемые проверки находятся в `src/data_checks.py`; формальный контракт полей описан в `src/fish_schema.json`.
5. В анализе есть паспорт полей, проверки формы/типов/пропусков/Inf/геометрии/категорий/повторов, семь графиков и сравнение выбросов IQR/MAD с индексами строк.
6. Проверены две гипотезы: связь массы с прокси-объёмом внутри видов и мультиколлинеарность трёх длин. В отчёте описано кодирование `Species` и обработка неизвестной категории.
7. До моделирования рассчитан групповой split 70/15/15 по округлённой геометрии; одинаковые измерения не пересекаются между наборами. Перечислены возможные каналы утечки и способы защиты.
8. Дополнительная часть: `python -m src.generate_html_report` строит `reports/eda_report.html`; `python -m unittest discover -s tests -v` проверяет контракт, в том числе смену масштаба веса и единицы в схеме.
9. Изменения пока находятся в рабочем дереве; коммиты, запрос на слияние и рецензирование ещё не выполнялись.

Паспорт данных
| Столбец | Смысл | Тип | Единицы | Назначение | Пропуски | Диапазон  | Проверки физического смысла |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Species** | Вид рыбы | категория | — | Категориальный признак | Нет | 7 видов: Bream, Roach, Whitefish, Parkki, Perch, Pike, Smelt | нет пустых строк и скрытых дубликатов написания |
| **Weight** | Масса рыбы | числовой | граммы (г) | **Целевая переменная** | Нет |  0 - 1650 | `Weight > 0`; обнаружено 1 нарушение (`Weight = 0`, Roach) |
| **Length1** | Длина от рыла до жаберной крышки | числовой | см | Признак | Нет | 7.5 … 59.0 | `> 0`; `Length1 < Length2 < Length3` |
| **Length2** | Длина от рыла до изгиба хвоста | числовой | см | Признак | Нет | 8.4 … 63.4 | `> 0`; `Length2 < Length3` |
| **Length3** | Длина от рыла до кончика хвоста | числовой | см | Признак | Нет | 8.8 … 68.0 | `> 0`; наибольшая из 3 длин |
| **Height** | Высота тела | числовой | см | Признак | Нет | 1.73 … 18.96 | `> 0`; в среднем меньше `Length3` |
| **Width** | Ширина тела | числовой | см | Признак | Нет | 1.05 … 8.14 | `> 0`; в среднем меньше `Length3` и `Height` |


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Ищем корень проекта независимо от текущей рабочей папки Jupyter.
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "assets" / "data" / "Fish.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Не найден assets/data/Fish.csv в корне проекта или родительских папках")

DATA_PATH = PROJECT_ROOT / "assets" / "data" / "Fish.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

assert not df.empty, "Датасет пуст!"
assert df.columns.is_unique, "Названия колонок дублируются!"

print(f"Размер таблицы: {df.shape[0]} строк, {df.shape[1]} столбцов")
print(f"Файл данных: {DATA_PATH}")
display(df.head())

quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(dropna=False),
})
display(quality)

Размер таблицы: 159 строк, 7 столбцов
Файл данных: d:\programming\ai\Fish-Dimensions-Regression-Analysis\assets\data\Fish.csv


,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


,dtype,missing,unique
Species,str,0,7
Weight,float64,0,101
Length1,float64,0,116
Length2,float64,0,93
Length3,float64,0,124
Height,float64,0,154
Width,float64,0,152


In [2]:
# ШАГ 3: Автоматизированный анализ качества данных

import sys
sys.path.insert(0, str(PROJECT_ROOT))
import src.data_checks as dc

print(f"Размерность: {dc.check_shape(df)}")
print(f"Полных дубликатов: {dc.check_duplicates(df)}")

print("\n=== ПОЛНЫЙ ОТЧЁТ О КАЧЕСТВЕ ДАННЫХ ===")
display(dc.full_quality_report(df))

print("\n=== ПРОПУСКИ, БЕСКОНЕЧНОСТИ И НЕВОЗМОЖНЫЕ ЗНАЧЕНИЯ ===")
print("Пропуски:", dc.check_missing(df).to_dict())
print("Бесконечности:", dc.check_infinity(df))
print("Неположительные значения:", dc.check_impossible(df))

print("\n=== ГЕОМЕТРИЧЕСКАЯ СОГЛАСОВАННОСТЬ ===")
print(dc.check_geometric_consistency(df))

print("\n=== РАСПРЕДЕЛЕНИЕ ВИДОВ ===")
display(dc.check_categories(df))

near_dups, _ = dc.find_near_duplicates(df)
print(f"\nСтрок в группах с совпадающей геометрией: {len(near_dups)}")
display(near_dups)

Размерность: (159, 7)
Полных дубликатов: 0

=== ПОЛНЫЙ ОТЧЁТ О КАЧЕСТВЕ ДАННЫХ ===


,column,dtype,missing,missing_share,unique,impossible
0,Species,str,0,0.0,7,0
1,Weight,float64,0,0.0,101,1
2,Length1,float64,0,0.0,116,0
3,Length2,float64,0,0.0,93,0
4,Length3,float64,0,0.0,124,0
5,Height,float64,0,0.0,154,0
6,Width,float64,0,0.0,152,0



=== ПРОПУСКИ, БЕСКОНЕЧНОСТИ И НЕВОЗМОЖНЫЕ ЗНАЧЕНИЯ ===
Пропуски: {'Species': 0, 'Weight': 0, 'Length1': 0, 'Length2': 0, 'Length3': 0, 'Height': 0, 'Width': 0}
Бесконечности: {'Weight': 0, 'Length1': 0, 'Length2': 0, 'Length3': 0, 'Height': 0, 'Width': 0}
Неположительные значения: {'Weight': 1, 'Length1': 0, 'Length2': 0, 'Length3': 0, 'Height': 0, 'Width': 0}

=== ГЕОМЕТРИЧЕСКАЯ СОГЛАСОВАННОСТЬ ===
{'Length1 > Length2': 0, 'Length2 > Length3': 0, 'Height >= Length3': 0, 'Width >= Length3': 0, 'any non-positive': 1}

=== РАСПРЕДЕЛЕНИЕ ВИДОВ ===


,count,share,is_rare
Species,,,
Perch,56,0.3522,False
Bream,35,0.2201,False
Roach,20,0.1258,False
Pike,17,0.1069,False
Smelt,14,0.0881,False
Parkki,11,0.0692,False
Whitefish,6,0.0377,True



Строк в группах с совпадающей геометрией: 2


,Species,Weight,Length1,Length2,Length3,Height,Width
142,Pike,1600.0,56.0,60.0,64.0,9.6,6.144
143,Pike,1550.0,56.0,60.0,64.0,9.6,6.144


In [3]:
# ШАГ 4: Построение и экспорт 7 графиков

from pathlib import Path
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import src.data_checks as dc

figures_dir = PROJECT_ROOT / "reports" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
df_plot = df.copy()
df_plot["log_Weight"] = np.log1p(df_plot["Weight"])
df_plot["Volume"] = df_plot["Length3"] * df_plot["Height"] * df_plot["Width"]

fig1 = make_subplots(rows=1, cols=2, subplot_titles=("Weight (г)", "log(1 + Weight)"))
fig1.add_trace(go.Histogram(x=df_plot["Weight"], nbinsx=25, marker_color="#1f77b4"), row=1, col=1)
fig1.add_trace(go.Histogram(x=df_plot["log_Weight"], nbinsx=25, marker_color="#2ca02c"), row=1, col=2)
fig1.update_layout(title="Распределение целевой переменной Weight", showlegend=False, width=900, height=400)
fig1.update_xaxes(title_text="Масса, г", row=1, col=1)
fig1.update_xaxes(title_text="log(1 + Weight)", row=1, col=2)
fig1.update_yaxes(title_text="Количество наблюдений", row=1, col=1)
fig1.write_html(figures_dir / "01_weight_dist.html")

species_counts = df_plot["Species"].value_counts().rename_axis("Species").reset_index(name="Count")
fig2 = make_subplots(rows=1, cols=2, subplot_titles=("Численность по видам (шт.)", "Weight по видам (г)"))
fig2.add_trace(go.Bar(x=species_counts["Species"], y=species_counts["Count"]), row=1, col=1)
for species in df_plot["Species"].unique():
    fig2.add_trace(go.Box(y=df_plot.loc[df_plot["Species"] == species, "Weight"], name=species), row=1, col=2)
fig2.update_layout(title="Численность и распределение массы по видам", width=950, height=450)
fig2.update_xaxes(title_text="Вид рыбы", row=1, col=1)
fig2.update_xaxes(title_text="Вид рыбы", row=1, col=2)
fig2.update_yaxes(title_text="Количество", row=1, col=1)
fig2.update_yaxes(title_text="Масса, г", row=1, col=2)
fig2.write_html(figures_dir / "02_weight_by_species.html")

fig3 = px.scatter_matrix(
    df_plot, dimensions=["Weight", "Length3", "Height", "Width"], color="Species",
    title="Матрица парных диаграмм", labels={"Weight": "Weight (г)", "Length3": "Length3 (см)"},
)
fig3.write_html(figures_dir / "03_pairplot.html")

corr = df[dc.NUMERIC_COLS].corr()
fig4 = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                 title="Корреляционная матрица Пирсона")
fig4.write_html(figures_dir / "04_corr.html")

quality_df = dc.full_quality_report(df)
fig5 = make_subplots(rows=1, cols=2, subplot_titles=("Доля пропусков", "Число уникальных значений"))
fig5.add_trace(go.Bar(x=quality_df["column"], y=quality_df["missing_share"]), row=1, col=1)
fig5.add_trace(go.Bar(x=quality_df["column"], y=quality_df["unique"]), row=1, col=2)
fig5.update_layout(title="Показатели качества данных", showlegend=False, width=900, height=400)
fig5.update_yaxes(title_text="Доля пропусков", range=[0, 1], row=1, col=1)
fig5.update_yaxes(title_text="Уникальные значения", row=1, col=2)
fig5.write_html(figures_dir / "05_quality.html")

bounds = dc.iqr_bounds(df_plot["Weight"])
fig6 = go.Figure(go.Box(x=df_plot["Weight"], name="Weight", boxpoints="all", jitter=0.3, pointpos=-1.8))
fig6.add_vline(x=bounds["upper"], line_dash="dash", line_color="red",
               annotation_text=f"Верхняя граница IQR ({bounds['upper']:.1f} г)")
fig6.update_layout(title="Выбросы массы по правилу IQR", xaxis_title="Масса, г", width=850, height=380)
fig6.write_html(figures_dir / "06_outliers.html")

fig7 = px.scatter(
    df_plot, x="Volume", y="Weight", color="Species", hover_data=["Length3", "Height", "Width"],
    title="Масса и прокси-объём по видам",
    labels={"Volume": "Приближённый объём (см³)", "Weight": "Масса (г)"},
)
fig7.write_html(figures_dir / "07_volume_weight.html")
print(f"Сохранены 7 интерактивных графиков: {figures_dir}")

Сохранены 7 интерактивных графиков: d:\programming\ai\Fish-Dimensions-Regression-Analysis\reports\figures


## Шаги 5–7: аномалии, гипотезы и защита от утечек

### IQR и MAD
IQR отмечает значения за пределами `Q1 - 1.5*IQR` и `Q3 + 1.5*IQR`; MAD использует модифицированный z-score с порогом `|z| > 3.5`. Статистический флаг не означает автоматическое удаление. Нулевой вес рассматривается отдельно как физическая ошибка; крупные щуки остаются в данных.

### Проверяемые гипотезы
1. Физический прокси-объём `Length3 × Height × Width` монотонно связан с весом внутри видов. Связь проверяется отдельно для каждого вида ранговой корреляцией Спирмена, чтобы не смешивать различия видов с общей связью.
2. Три линейных измерения длины сильно коррелируют между собой. Проверяется матрица корреляций Пирсона; корреляция сама по себе не доказывает причинность и не является автоматическим основанием удалить поле.

Для `Species` при моделировании подходит one-hot encoding. Неизвестный вид следует обрабатывать `handle_unknown="ignore"` либо отклонять явной проверкой схемы; молча переобучать кодировщик на тестовых данных нельзя.

### Правило разбиения и каналы утечки
Разбивать на train/validation/test в пропорции около 70/15/15 по группам одинаковой геометрии (округление измерений до 2 знаков), фиксируя seed. Все строки одной группы должны попасть только в один набор. Из-за малого числа `Whitefish` обычная стратификация не гарантирует стабильную оценку; состав видов в каждом наборе нужно явно проверять. Все преобразования и подбор признаков настраиваются только на train.

| Канал утечки | Риск | Защита |
| --- | --- | --- |
| Статистики всей таблицы | Масштабирование/импутация использует validation/test | Fit только на train |
| Повторные геометрические строки | Почти одинаковые объекты оказываются в train и test | Групповой split по геометрии |
| Идентификатор или индекс CSV | Модель запоминает порядок/объект | Не включать индекс как признак |
| Измерение после взвешивания | Признак содержит информацию о целевой переменной | Проверить момент и протокол измерения у источника |
| Ручной выбор по test | Тест перестаёт быть независимой оценкой | Зафиксировать test и не использовать для выбора модели |
| Подбор признаков по всей таблице | Переоценка качества из-за использования test | Отбор выполнять внутри train/CV |



In [4]:
# Шаг 5: IQR/MAD-флаги с исходными индексами
weight = df["Weight"]
iqr_mask, iqr = dc.outliers_iqr(weight)
mad_mask, mad_scores, median_weight, mad = dc.outliers_mad(weight)

anomalies = df.loc[iqr_mask | mad_mask].copy()
anomalies.insert(0, "source_index", anomalies.index)
anomalies["iqr_flag"] = iqr_mask.loc[anomalies.index].astype(int)
anomalies["mad_flag"] = mad_mask.loc[anomalies.index].astype(int)
print(f"IQR bounds: [{iqr['lower']:.1f}, {iqr['upper']:.1f}] г")
print(f"MAD median={median_weight:.1f} г, MAD={mad:.1f} г")
print("Подозрительные строки по IQR или MAD:")
display(anomalies[["source_index", "Species", "Weight", "iqr_flag", "mad_flag"]])

checked = df.copy()
checked["flag_error"] = (checked["Weight"] <= 0).astype(int)
checked["flag_outlier"] = (iqr_mask | mad_mask).astype(int)
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
checked.to_csv(processed_dir / "fish_checked.csv", index=False, encoding="utf-8-sig")

# Гипотеза 1: физический прокси-объём связан с массой внутри вида.
df_hypothesis = df.assign(Volume=df["Length3"] * df["Height"] * df["Width"])
volume_correlations = df_hypothesis.groupby("Species", observed=True).apply(
    lambda group: group[["Volume", "Weight"]].corr(method="spearman").iloc[0, 1],
    include_groups=False,
).rename("spearman_volume_weight").sort_values(ascending=False)
print("Гипотеза 1: Spearman(Volume, Weight) отдельно по видам")
display(volume_correlations.to_frame())

# Гипотеза 2: три измерения длины несут сильно коррелированную информацию.
length_corr = df[["Length1", "Length2", "Length3"]].corr()
print("Гипотеза 2: корреляции Пирсона между длинами")
display(length_corr)

# Групповой split 70/15/15 по геометрии; одинаковые измерения не разделяются.
geometry = ["Length1", "Length2", "Length3", "Height", "Width"]
group_keys = df[geometry].round(2).astype(str).agg("|".join, axis=1)
unique_groups = np.array(group_keys.unique())
shuffled_groups = np.random.default_rng(42).permutation(unique_groups)
train_end = int(len(shuffled_groups) * 0.70)
validation_end = int(len(shuffled_groups) * 0.85)
split_by_group = {
    "train": set(shuffled_groups[:train_end]),
    "validation": set(shuffled_groups[train_end:validation_end]),
    "test": set(shuffled_groups[validation_end:]),
}
assert not (split_by_group["train"] & split_by_group["validation"])
assert not (split_by_group["train"] & split_by_group["test"])
assert not (split_by_group["validation"] & split_by_group["test"])
for split_name, split_groups in split_by_group.items():
    split_rows = df.loc[group_keys.isin(split_groups)]
    print(f"{split_name}: {len(split_rows)} строк; виды: {split_rows['Species'].value_counts().to_dict()}")

IQR bounds: [-675.0, 1445.0] г
MAD median=273.0 г, MAD=204.0 г
Подозрительные строки по IQR или MAD:


,source_index,Species,Weight,iqr_flag,mad_flag
142,142,Pike,1600.0,1,1
143,143,Pike,1550.0,1,1
144,144,Pike,1650.0,1,1


Гипотеза 1: Spearman(Volume, Weight) отдельно по видам


,spearman_volume_weight
Species,
Parkki,0.990909
Perch,0.986000
Whitefish,0.985611
Pike,0.982177
Smelt,0.966963
Bream,0.948889
Roach,0.945072


Гипотеза 2: корреляции Пирсона между длинами


,Length1,Length2,Length3
Length1,1.000000,0.999517,0.992031
Length2,0.999517,1.000000,0.994103
Length3,0.992031,0.994103,1.000000


train: 111 строк; виды: {'Perch': 45, 'Bream': 22, 'Roach': 13, 'Pike': 11, 'Smelt': 9, 'Parkki': 8, 'Whitefish': 3}
validation: 24 строк; виды: {'Bream': 7, 'Perch': 6, 'Smelt': 3, 'Roach': 2, 'Whitefish': 2, 'Parkki': 2, 'Pike': 2}
test: 24 строк; виды: {'Bream': 6, 'Roach': 5, 'Perch': 5, 'Pike': 4, 'Smelt': 2, 'Whitefish': 1, 'Parkki': 1}


## Дополнительное задание: формальная схема и HTML

Строки проверяются по JSON-контракту `src/fish_schema.json`. Финальная ячейка проверяет схему и генерирует единый HTML-отчёт со сводкой качества, аномалиями и ссылками на графики. Независимые проверки изменения единиц и диапазонов запускаются командой `python -m unittest discover -s tests -v`.

In [5]:
import importlib
import src.data_checks as dc
dc = importlib.reload(dc)
schema_errors = dc.validate_fish_schema(df)
assert not schema_errors, schema_errors
from src.generate_html_report import generate_report
print(f"Автоматический HTML-отчёт: {generate_report()}")

Автоматический HTML-отчёт: D:\programming\ai\Fish-Dimensions-Regression-Analysis\reports\eda_report.html
